# Lab 2 · Exercise C — Pulls, bias and coverage

Test your fit with **toy experiments**: generate with known truth, fit, and calculate the **pull** for the signal yield: 

$$ \text{pull} = \frac{\hat{n}_{sig} - n_{sig}^{true}}{\sigma_{\hat{n}_{sig}}} $$

First, set the number of toys to a low number to test the script, then increase.

In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
m     = ROOT.RooRealVar("m", "m [MeV]", 5080, 5540)
mean  = ROOT.RooRealVar("mean", "mean", 5288, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 14.8, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = ROOT.RooGaussian("sig", "sig", m, mean, sigma)
bkg   = ROOT.RooExponential("bkg", "bkg", m, slope)
nsig  = ROOT.RooRealVar("nsig", "signal yield", 33, 0, 400)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 133, 0, 600)
model = ROOT.RooAddPdf("model", "sig+bkg", ROOT.RooArgList(sig, bkg), ROOT.RooArgList(nsig, nbkg))

## C1 · The toy loop
Complete `run_toys`: for each toy, **reset** the parameters to the truth, **generate** an extended dataset from the model,
**fit** it, and record the pull of `nsig` if the fit converged (`status() == 0`).

In [ ]:

# 1. Set true values for the toy generation
nsig_true = 33.0
nbkg_true = 140.0 # Set the true background yield (adjust to your model)

nsig.setVal(nsig_true)
nbkg.setVal(nbkg_true)

# 2. Initialize RooMCStudy
# We pass the model, the observable, and specify Extended() for generation.
# FitOptions(Save(True)) is CRUCIAL: it keeps the results of each toy fit in memory.
print("Initializing RooMCStudy...")
mcs = ROOT.RooMCStudy(
    model, 
    ROOT.RooArgSet(m), 
    ROOT.RooFit.Extended(), 
    ROOT.RooFit.FitOptions(ROOT.RooFit.Save(True))
)

In [ ]:

# 3. Generate and fit the toys
# RooMCStudy handles the loop, the memory management, and the fit status checks internally!
n_toys = 10  # 500
print(f"Generating and fitting {n_toys} toys in C++...")
mcs.generateAndFit(n_toys)
print("Done.")

In [ ]:
# 4. Plot the pull distribution using native RooFit plotting!
# plotPull creates a RooPlot with the pull histogram and a Gaussian fit overlay automatically.
pull_plot = mcs.plotPull(nsig)

c_pull = ROOT.TCanvas("c_pull", "Pull Distribution", 800, 600)
pull_plot.Draw()
c_pull.Draw()

In [ ]:
# 5. Extract the statistics
# RooMCStudy automatically calculates the mean and RMS of the pull distribution
# ==> pullDist may have isseus in python bindin
#  
# h_pull = mcs.pullDist(nsig)
# print(f"\n--- Pull Statistics for nsig ---")
# print(f"Pull Mean: {h_pull.GetMean():.3f} (Expected ~0)")
# print(f"Pull RMS:  {h_pull.GetRMS():.3f} (Expected ~1)")

# Get the dataset containing the results of all toy experiments
dataset = mcs.fitParDataSet()

# Print the dataset to verify the exact names of the automatically generated pull variables
# dataset.Print() 

# Retrieve the specific pull variable using its generated name
# (e.g., "nsigpull", "meanpull", "sigmapull")
pull_var = dataset.get().find("nsigpull")

# Extract the Mean and RMS directly from the dataset
mean_pull = dataset.mean(pull_var)
rms_pull = dataset.sigma(pull_var)

print(f"Pull Mean: {mean_pull:.4f}")
print(f"Pull RMS:  {rms_pull:.4f}")

## C2 · Low statistics
Repeat with $\nu_s=5$. Look at the pull distribution, the fraction of fits with `nsig` at the lower boundary (nsig < 0.01) and the coverage.

In [ ]:

# TODO: print mean, RMS, coverage |pull|<1, failed fits, and the fraction of toys with nsig < 0.01
# TODO: histogram of the pulls

## Questions (optional)
Write your answer under each question, in 2–3 lines, and quote the numbers or plots of the steps indicated.
1. **(C1)** For $\nu_s=33$: is the fit biased (look at the mean of the pulls)? Are the HESSE errors reliable (look at their width)? Write the two numbers.
2. **(C2)** For $\nu_s=5$: what is different in the pull distribution compared with C1 (mean, width, shape), and why? What would you report instead of a symmetric error?
3. **(C1, C2)** In the toy loop some fits fail. Why is it important to count them? What bias do you introduce if you discard them from the pulls?